In [1]:
import logging
import mimetypes
import os
import time
from tqdm import tqdm
from argparse import ArgumentParser
import decord 

import cv2
import json_tricks as json
import mmcv
import mmengine
import numpy as np
from mmengine.logging import print_log

from mmpose.apis import inference_topdown
from mmpose.apis import init_model as init_pose_estimator
from mmpose.registry import VISUALIZERS
from mmpose.structures import merge_data_samples, split_instances

from multiprocessing import Pool, cpu_count

import logging
# try:
    # from mmdet.apis import inference_detector, init_detector
    # has_mmdet = True
# except (ImportError, ModuleNotFoundError):
    # has_mmdet = False


decord.bridge.set_bridge('native')
logging.basicConfig(level=logging.INFO)


In [2]:
def process_one_image(img,
                      bboxe,
                      pose_estimator):
    """Visualize predicted keypoints (and heatmaps) of one image."""

    # predict bbox
    # det_result = inference_detector(detector, img)
    # pred_instance = det_result.pred_instances.cpu().numpy()
    # bboxes = np.concatenate(
        # (pred_instance.bboxes, pred_instance.scores[:, None]), axis=1)
    # bboxes = bboxes[np.logical_and(pred_instance.labels == args.det_cat_id,
                                #    pred_instance.scores > args.bbox_thr)]
    # bboxes = bboxes[nms(bboxes, args.nms_thr), :4]
    
    height, width, _ = img.shape
    # bbox_index_mapping = {i: key for i, key in enumerate(bboxe.keys())}
    bboxes = []
    bboxs_label = []
    for key in bboxe.keys():
        bboxes.append([int(bboxe[key][0]*width), int(bboxe[key][1]*height), int(bboxe[key][2]*width), int(bboxe[key][3]*height)])
        bboxs_label.append(key)
    bboxes = np.array(bboxes, dtype=np.int16)
    bboxs_label = np.array(bboxs_label, dtype=np.int16)
    # predict keypoints
    pose_results = inference_topdown(pose_estimator, img, bboxes)
    data_samples = merge_data_samples(pose_results)

    # add keypoints label 
    pred_inst = data_samples.get('pred_instances', None)

    # Sometime we don't have bbox but the model still put one and it needs to be handle
    if len(bboxs_label) != len(pred_inst.get('bbox_scores')):
        bboxs_label = np.ones(pred_inst.get('bbox_scores').shape) *-1    
        
    pred_inst.set_data({'keypoints_label': bboxs_label})

    # if there is no instance detected, return None
    return data_samples.get('pred_instances', None)

def init_worker(pose_config, pose_checkpoint, device):
    """Initialize worker with a deep learning model (one per process)."""
    global pose_estimator
    pose_estimator = init_pose_estimator(pose_config, pose_checkpoint, device=device)


def process_frame_chunk(id_fr_bbox:list):
    # topdown pose estimation
    global pose_estimator

    logging.info(f'Start chunk {id_fr_bbox[0][0]}')
    pred_instances_chunk = []

    for frame_idx, fr, bbox in id_fr_bbox:
        fr = cv2.cvtColor(fr, cv2.COLOR_RGB2BGR)
        pred_instances = process_one_image(fr, bbox,
                                        pose_estimator)
        
        pred_instances_chunk.append(
        dict(
            frame_id=frame_idx,
            instances=split_instances(pred_instances)))
        
    logging.info(f"{frame_idx} done")
        
    return pred_instances_chunk

In [3]:
def main(line_argument=None):
    """Visualize the demo images.

    Using mmdet to detect the human.
    """
    parser = ArgumentParser()
    parser.add_argument('pose_config', help='Config file for pose')
    parser.add_argument('pose_checkpoint', help='Checkpoint file for pose')
    parser.add_argument(
        '--input', type=str, default='', help='Image/Video file')
    parser.add_argument(
        '--output-root',
        type=str,
        default='',
        help='root of the output img file. '
        'Default not saving the visualization images.')

    parser.add_argument(
        '--device', default='cuda:0', help='Device used for inference')

    parser.add_argument(
        '--skeleton-style',
        default='mmpose',
        type=str,
        choices=['mmpose', 'openpose'],
        help='Skeleton style selection')
    
    parser.add_argument(
        '--bboxes', type=str, help='Path to the json file which contains the bbox')
    
    parser.add_argument('--batch-size', default=32, type=int)

    # assert has_mmdet, 'Please install mmdet to run the demo.'

    args = parser.parse_args(line_argument)

    logging.info(f"Params : {args.__dict__}")

    assert (args.output_root != '')
    assert args.input != ''
    output_file = None

    mmengine.mkdir_or_exist(args.output_root)


    assert args.output_root != ''
    args.pred_save_path = f'{args.output_root}/results_' \
        f'{os.path.splitext(os.path.basename(args.input))[0]}.json'


    with open(args.bboxes, 'r') as fd:
        bboxes = json.load(fd)

    

    vr = decord.VideoReader(args.input)

    frame_indices = list(range(len(vr)))
    frame_chunks = [frame_indices[i:i + args.batch_size] for i in range(0, len(frame_indices), args.batch_size)]

    pred_instances_list = []

    

    logging.info('Start Pool')
    # num_workers = cpu_count()
    num_workers = 1
    with Pool(num_workers, initializer=init_worker, initargs=(args.pose_config, args.pose_checkpoint, args.device)) as pool:

        for chunk in frame_chunks:
            frames = [(idx, vr[idx].asnumpy(), bboxes[str(idx)]) for idx in chunk]  # Load frames lazily
            # pred_instances_chunk = pool.apply_async(process_frame_chunk, (frames,))
            pred_instances_chunk = pool.apply(process_frame_chunk, (frames,))
            pred_instances_list.extend(pred_instances_chunk.get())

            pool.close()

    logging.info('Frame processing finished')
    
    with open(args.pred_save_path, 'w') as f:
        json.dump(
            dict(
                meta_info=pose_estimator.dataset_meta,
                instance_info=pred_instances_list),
            f,
            indent='\t')
    logging.info(f'predictions have been saved at {args.pred_save_path}')

    input_type = input_type.replace('webcam', 'video')
    logging.info_log(
        f'the output {input_type} has been saved at {output_file}',
        logger='current',
        level=logging.INFO)



In [4]:
arg=[
    r"D:\idtracking\vitpose_sam2\mmpose\configs\body_2d_keypoint\topdown_heatmap\coco\td-hm_ViTPose-huge_8xb64-210e_coco-256x192.py",
     r"D:\idtracking\weights_config\td-hm_ViTPose-huge_8xb64-210e_coco-256x192-e32adcd4_20230314.pth",
    "--input", r"D:\idtracking\data\video\7965_T2a_ADOS_25fps.mp4",
    "--bboxes", r"D:\idtracking\data\video\7965_T2a_ADOS_25fps\bbox.json",
    "--output-root", r"D:\idtracking\data\video\7965_T2a_ADOS_25fps_2",
    "--device", "cuda",
    "--skeleton-style", "openpose",
    "--batch-size", "2"
]


In [ ]:
main(arg)

# TODO mapping bbox with id from sam2

INFO:root:Params : {'pose_config': 'D:\\idtracking\\vitpose_sam2\\mmpose\\configs\\body_2d_keypoint\\topdown_heatmap\\coco\\td-hm_ViTPose-huge_8xb64-210e_coco-256x192.py', 'pose_checkpoint': 'D:\\idtracking\\weights_config\\td-hm_ViTPose-huge_8xb64-210e_coco-256x192-e32adcd4_20230314.pth', 'input': 'D:\\idtracking\\data\\video\\7965_T2a_ADOS_25fps.mp4', 'output_root': 'D:\\idtracking\\data\\video\\7965_T2a_ADOS_25fps_2', 'device': 'cuda', 'skeleton_style': 'openpose', 'bboxes': 'D:\\idtracking\\data\\video\\7965_T2a_ADOS_25fps\\bbox.json', 'batch_size': 2}
INFO:root:Start Pool
